In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric_df["timestamp_dt"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
log_df["timestamp_dt"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)
trace_df["timestamp_dt"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True)
error_logs_df["timestamp_dt"] = pd.to_datetime(error_logs_df["timestamp"], unit="s", utc=True)

# Helper to get unique cmdb_id list limited to first 50
def sample_unique(series, limit=50):
    vals = pd.Series(series.dropna().unique())
    return vals.iloc[:limit].tolist()

# Metric summary
metric_summary = pd.DataFrame([{
    "file": "metric.csv",
    "rows": metric_df.shape[0],
    "min_timestamp_utc": metric_df["timestamp_dt"].min(),
    "max_timestamp_utc": metric_df["timestamp_dt"].max(),
    "unique_cmdb_ids_sample": sample_unique(metric_df["cmdb_id"], 50)
}])

# Metric KPI info: number of unique kpi_name and up to first 50 distinct names
metric_kpi_info = pd.DataFrame([{
    "unique_kpi_count": int(metric_df["kpi_name"].nunique(dropna=True)),
    "kpi_name_sample": metric_df["kpi_name"].dropna().unique()[:50].tolist()
}])

# Log summary
log_summary = pd.DataFrame([{
    "file": "log.csv",
    "rows": log_df.shape[0],
    "min_timestamp_utc": log_df["timestamp_dt"].min(),
    "max_timestamp_utc": log_df["timestamp_dt"].max(),
    "unique_cmdb_ids_sample": sample_unique(log_df["cmdb_id"], 50)
}])

# Log name info
log_name_info = pd.DataFrame([{
    "unique_log_name_count": int(log_df["log_name"].nunique(dropna=True)) if "log_name" in log_df.columns else 0,
    "log_name_sample": log_df["log_name"].dropna().unique()[:50].tolist() if "log_name" in log_df.columns else []
}])

# Trace summary
trace_summary = pd.DataFrame([{
    "file": "trace.csv",
    "rows": trace_df.shape[0],
    "min_timestamp_utc": trace_df["timestamp_dt"].min(),
    "max_timestamp_utc": trace_df["timestamp_dt"].max(),
    "unique_cmdb_ids_sample": sample_unique(trace_df["cmdb_id"], 50)
}])

# Trace name info
trace_name_info = pd.DataFrame([{
    "unique_trace_name_count": int(trace_df["trace_name"].nunique(dropna=True)) if "trace_name" in trace_df.columns else 0,
    "trace_name_sample": trace_df["trace_name"].dropna().unique()[:50].tolist() if "trace_name" in trace_df.columns else []
}])

# Error logs: prepare sample with truncated message (first 200 chars), limit to first 20 rows
error_logs_df["message_trunc"] = error_logs_df.get("message", pd.Series([""] * len(error_logs_df))).fillna("").astype(str).str.slice(0,200)
error_logs_sample = error_logs_df.loc[:, ["timestamp_dt", "cmdb_id", "message_trunc"]].head(20)

# Display compact summaries and samples (use variable names, not print)
metric_summary, metric_kpi_info, log_summary, log_name_info, trace_summary, trace_name_info, error_logs_sample

```
Out[1]:
```
```python
# Summarize results in plain English using previous variables from the kernel
summary = (
    f"metric.csv: {int(metric_summary.at[0,'rows'])} rows, time range "
    f"{metric_summary.at[0,'min_timestamp_utc'].isoformat()} to {metric_summary.at[0,'max_timestamp_utc'].isoformat()}. "
    f"Sample cmdb_ids: {', '.join(metric_summary.at[0,'unique_cmdb_ids_sample'][:10])} "
    f"(+ more). Unique kpi_name count: {int(metric_kpi_info.at[0,'unique_kpi_count'])}. "
    f"Sample kpi_names: {', '.join(metric_kpi_info.at[0,'kpi_name_sample'][:10])}."
    "\n\n"
    f"log.csv: {int(log_summary.at[0,'rows'])} rows, time range "
    f"{log_summary.at[0,'min_timestamp_utc'].isoformat()} to {log_summary.at[0,'max_timestamp_utc'].isoformat()}. "
    f"Sample cmdb_ids: {', '.join(log_summary.at[0,'unique_cmdb_ids_sample'][:10])} "
    f"(+ more). Unique log_name count: {int(log_name_info.at[0,'unique_log_name_count'])}. "
    f"Sample log_names: {', '.join(log_name_info.at[0,'log_name_sample'][:10])}."
    "\n\n"
    f"trace.csv: {int(trace_summary.at[0,'rows'])} rows. No timestamps or trace entries present. "
    f"Unique trace_name count: {int(trace_name_info.at[0,'unique_trace_name_count'])}."
    "\n\n"
    f"error_logs.csv: {error_logs_sample.shape[0]} rows in the displayed sample (first 20). "
    f"No sample rows to show." if error_logs_sample.shape[0]==0 else
    (f"Sample error log rows shown (up to 20).")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  rows         min_timestamp_utc         max_timestamp_utc                             unique_cmdb_ids_sample
0  metric.csv  1900 2024-01-20 12:17:00+00:00 2024-01-20 12:41:00+00:00  [carts, carts-db, catalogue, catalogue-db, fro...,    unique_kpi_count                                    kpi_name_sample
0                 8  [cpu, latency-50, latency-90, mem, socket, wor...,       file  rows         min_timestamp_utc         max_timestamp_utc                             unique_cmdb_ids_sample
0  log.csv   454 2024-01-20 12:17:00+00:00 2024-01-20 12:41:00+00:00  [carts, carts-db, catalogue, front-end, orders...,    unique_log_name_count                   log_name_sample
0                      2  [log.error_count, log.row_count],         file  rows min_timestamp_utc max_timestamp_utc unique_cmdb_ids_sample
0  trace.csv     0               NaT               NaT                     [],    unique_trace_name_count trace_name_sample
0                        0                [], Empty DataFrame
Columns: [timestamp_dt, cmdb_id, message_trunc]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df["timestamp_dt"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)

# 1) Calculate global P95 per (cmdb_id, kpi_name) using the full series (global threshold)
p95_df = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], dropna=False)["value"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"value": "p95_threshold"})
)

# 2) Filter metric data to the incident window (inclusive)
start_ts = pd.to_datetime("2024-01-20 12:14:22", utc=True)
end_ts   = pd.to_datetime("2024-01-20 12:44:22", utc=True)
window_df = metric_df[(metric_df["timestamp_dt"] >= start_ts) & (metric_df["timestamp_dt"] <= end_ts)].copy()

# 3) Merge p95 thresholds into the window data (right join to keep all component-KPI combos scanned)
merged = window_df.merge(p95_df, on=["cmdb_id", "kpi_name"], how="right", validate="many_to_one", suffixes=("", "_p95"))

# 4) Determine anomaly flag per row in the window (value >= p95_threshold)
merged["is_anomaly"] = merged["value"].ge(merged["p95_threshold"]).fillna(False)

# 5) Aggregate per component-KPI restricted to the window
agg = merged.groupby(["cmdb_id", "kpi_name"], dropna=False).agg(
    p95_threshold=("p95_threshold", "first"),
    rows_in_window=("value", "count"),  # number of rows present in the window for this combo
    count_anomalies=("is_anomaly", "sum"),
    earliest_anomaly_timestamp=("timestamp_dt", lambda s: s[merged.loc[s.index, "is_anomaly"]].min() if s.notna().any() else pd.NaT),
    max_value_in_window=("value", "max"),
).reset_index()

# 6) Compute severity_ratio (max_value_in_window / p95_threshold) with safe divide
agg["severity_ratio"] = agg.apply(
    lambda r: (r["max_value_in_window"] / r["p95_threshold"]) if pd.notna(r["p95_threshold"]) and r["p95_threshold"] != 0 and pd.notna(r["max_value_in_window"]) else np.nan,
    axis=1
)

# 7) Compact result: top 20 by count_anomalies desc, then severity_ratio desc
agg_sorted = agg.sort_values(["count_anomalies", "severity_ratio"], ascending=[False, False]).reset_index(drop=True)
result_top20 = agg_sorted.loc[:, ["cmdb_id", "kpi_name", "p95_threshold", "count_anomalies", "earliest_anomaly_timestamp", "max_value_in_window", "severity_ratio"]].head(20)

# 8) Additional requested summaries
total_component_kpi_combos = int(p95_df.shape[0])
distinct_kpi_names = p95_df["kpi_name"].dropna().unique()[:50].tolist()
distinct_cmdb_ids = p95_df["cmdb_id"].dropna().unique()[:50].tolist()

# 9) If no anomalies found (all count_anomalies == 0) return empty table indicator
if (agg["count_anomalies"] > 0).sum() == 0:
    # empty result table
    result_top20 = result_top20.iloc[0:0]  # empty DataFrame with same columns
    no_anoms_msg = "No anomalies found in the incident window (no values >= global P95 thresholds)."
else:
    no_anoms_msg = ""

# Display compact outputs (variables)
result_top20, {"total_component_kpi_combos": total_component_kpi_combos,
               "distinct_kpi_names_sample": distinct_kpi_names,
               "distinct_cmdb_ids_sample": distinct_cmdb_ids,
               "note": no_anoms_msg}

```
Out[2]:
```
Summary of results (computed P95 from full series; incident window 2024-01-20 12:14:22 → 2024-01-20 12:44:22 UTC):

- Total distinct component-KPI combos scanned: 76
- Distinct kpi_name sample (up to 50): cpu, latency-50, latency-90, mem, socket, workload, diskio, error
- Distinct cmdb_id sample (up to 50): carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db
- Anomalies found: Yes (rows where value >= global P95 exist)

Top 20 component-KPI combos ranked by count_anomalies (count_anomalies, earliest anomaly timestamp, p95_threshold, max_value_in_window, severity_ratio):

1) carts-db / socket — p95=6.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=6.0, severity=1.00  
2) catalogue / socket — p95=5.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=5.0, severity=1.00  
3) catalogue-db / socket — p95=4.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=4.0, severity=1.00  
4) orders-db / socket — p95=7.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=7.0, severity=1.00  
5) queue-master / socket — p95=3.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=3.0, severity=1.00  
6) rabbitmq / socket — p95=11.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=11.0, severity=1.00  
7) rabbitmq-exporter / mem — p95=4,546,560.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=4,546,560.0, severity=1.00  
8) rabbitmq-exporter / socket — p95=1.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=1.0, severity=1.00  
9) session-db / socket — p95=3.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=3.0, severity=1.00  
10) catalogue-db / diskio — p95=0.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=0.0, severity=NaN (p95 = 0)  
11) front-end / error — p95=0.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=0.0, severity=NaN (p95 = 0)  
12) orders / error — p95=0.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=0.0, severity=NaN (p95 = 0)  
13) rabbitmq / diskio — p95=0.0, anomalies=25, earliest=2024-01-20T12:17:00+00:00, max=0.0, severity=NaN (p95 = 0)  
14) shipping / socket — p95=6.0, anomalies=21, earliest=2024-01-20T12:17:00+00:00, max≈6.131579, severity≈1.02193  
15) payment / socket — p95=6.0, anomalies=12, earliest=2024-01-20T12:30:00+00:00, max=6.0, severity=1.00  
16) payment / diskio — p95≈26,391.62, anomalies=2, earliest=2024-01-20T12:17:00+00:00, max≈69,891.94, severity≈2.64826  
17) orders / cpu — p95≈1.883249, anomalies=2, earliest=2024-01-20T12:28:00+00:00, max≈4.334584, severity≈2.30165  
18) rabbitmq-exporter / cpu — p95≈0.000129189, anomalies=2, earliest=2024-01-20T12:20:00+00:00, max≈0.000187446, severity≈1.45094  
19) shipping / cpu — p95≈1.194182, anomalies=2, earliest=2024-01-20T12:19:00+00:00, max≈1.578741, severity≈1.32203  
20) carts / cpu — p95≈2.268066, anomalies=2, earliest=2024-01-20T12:23:00+00:00, max≈2.963401, severity≈1.30658

Notes / interpretation:
- Many top anomalies are for the "socket" metric across multiple services (several with exact p95 == max in-window → severity 1.0). This indicates socket values reached their historical 95th percentile during the incident window at many components.
- A few metrics have p95 == 0 (diskio, error for some services); severity is undefined (NaN) in those cases because division by zero is not meaningful — these should be inspected separately (zero baseline).
- Highest severity ratios (notable spikes) in the top 20: payment.diskio (~2.65), orders.cpu (~2.30), shipping.socket (~1.02) and several CPU entries with >1.3.

If you want, I can:
- expand the table to show more details,
- produce per-kpi or per-service summaries,
- or list only combinations with severity_ratio > 1.5.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  p95_threshold  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0            carts-db   socket   6.000000e+00               25  2024-01-20 12:17:00+00:00         6.000000e+00        1.000000
1           catalogue   socket   5.000000e+00               25  2024-01-20 12:17:00+00:00         5.000000e+00        1.000000
2        catalogue-db   socket   4.000000e+00               25  2024-01-20 12:17:00+00:00         4.000000e+00        1.000000
3           orders-db   socket   7.000000e+00               25  2024-01-20 12:17:00+00:00         7.000000e+00        1.000000
4        queue-master   socket   3.000000e+00               25  2024-01-20 12:17:00+00:00         3.000000e+00        1.000000
5            rabbitmq   socket   1.100000e+01               25  2024-01-20 12:17:00+00:00         1.100000e+01        1.000000
6   rabbitmq-exporter      mem   4.546560e+06               25  2024-01-20 12:17:00+00:00         4.546560e+06        1.000000
7   rabbitmq-exporter   socket   1.000000e+00               25  2024-01-20 12:17:00+00:00         1.000000e+00        1.000000
8          session-db   socket   3.000000e+00               25  2024-01-20 12:17:00+00:00         3.000000e+00        1.000000
9        catalogue-db   diskio   0.000000e+00               25  2024-01-20 12:17:00+00:00         0.000000e+00             NaN
10          front-end    error   0.000000e+00               25  2024-01-20 12:17:00+00:00         0.000000e+00             NaN
11             orders    error   0.000000e+00               25  2024-01-20 12:17:00+00:00         0.000000e+00             NaN
12           rabbitmq   diskio   0.000000e+00               25  2024-01-20 12:17:00+00:00         0.000000e+00             NaN
13           shipping   socket   6.000000e+00               21  2024-01-20 12:17:00+00:00         6.131579e+00        1.021930
14            payment   socket   6.000000e+00               12  2024-01-20 12:30:00+00:00         6.000000e+00        1.000000
15            payment   diskio   2.639162e+04                2  2024-01-20 12:17:00+00:00         6.989194e+04        2.648262
16             orders      cpu   1.883249e+00                2  2024-01-20 12:28:00+00:00         4.334584e+00        2.301652
17  rabbitmq-exporter      cpu   1.291890e-04                2  2024-01-20 12:20:00+00:00         1.874459e-04        1.450943
18           shipping      cpu   1.194182e+00                2  2024-01-20 12:19:00+00:00         1.578741e+00        1.322027
19              carts      cpu   2.268066e+00                2  2024-01-20 12:23:00+00:00         2.963401e+00        1.306576, {'total_component_kpi_combos': 76, 'distinct_kpi_names_sample': ['cpu', 'latency-50', 'latency-90', 'mem', 'socket', 'workload', 'diskio', 'error'], 'distinct_cmdb_ids_sample': ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'], 'note': ''})```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if available, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamp columns parsed to UTC datetimes
if "timestamp_dt" not in log_df.columns or not pd.api.types.is_datetime64_any_dtype(log_df["timestamp_dt"]):
    log_df["timestamp_dt"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)
if "timestamp_dt" not in error_logs_df.columns or not pd.api.types.is_datetime64_any_dtype(error_logs_df["timestamp_dt"]):
    error_logs_df["timestamp_dt"] = pd.to_datetime(error_logs_df["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 12:14:22", utc=True)
end_ts   = pd.to_datetime("2024-01-20 12:44:22", utc=True)

# --- Part 1: log.csv analysis ---

# 1.1 Global P95 per (cmdb_id, log_name) using full series
log_p95 = (
    log_df
    .groupby(["cmdb_id", "log_name"], dropna=False)["value"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"value": "p95_threshold"})
)

# 1.2 Filter log rows to the incident window
log_window = log_df[(log_df["timestamp_dt"] >= start_ts) & (log_df["timestamp_dt"] <= end_ts)].copy()

# 1.3 Merge thresholds with window data (keep all combos scanned)
merged_log = log_window.merge(log_p95, on=["cmdb_id", "log_name"], how="right", validate="many_to_one")

# 1.4 Determine anomaly flag (value >= p95_threshold) for rows in the window; non-window rows will have NaN value -> not anomaly
merged_log["is_anomaly"] = merged_log["value"].ge(merged_log["p95_threshold"]).fillna(False)

# 1.5 Aggregate per (cmdb_id, log_name) restricted to the window (but using global p95_threshold)
grp = merged_log.groupby(["cmdb_id", "log_name"], dropna=False)

p95_series = grp["p95_threshold"].first()
rows_in_window = grp["value"].count()
count_anomalies = grp["is_anomaly"].sum()
earliest_anomaly = grp.apply(lambda g: g.loc[g["is_anomaly"], "timestamp_dt"].min() if g["is_anomaly"].any() else pd.NaT)
max_value = grp["value"].max()

log_agg = (
    pd.DataFrame({
        "p95_threshold": p95_series,
        "rows_in_window": rows_in_window,
        "count_anomalies": count_anomalies,
        "earliest_anomaly_timestamp": earliest_anomaly,
        "max_value_in_window": max_value
    })
    .reset_index()
)

# 1.6 severity_ratio (handle p95==0 or NaN -> np.nan)
def compute_severity(row):
    p95 = row["p95_threshold"]
    mx = row["max_value_in_window"]
    if pd.isna(p95) or p95 == 0 or pd.isna(mx):
        return np.nan
    return mx / p95

log_agg["severity_ratio"] = log_agg.apply(compute_severity, axis=1)

# 1.7 Select and sort top 20 by count_anomalies desc then severity desc
log_result_top20 = (
    log_agg
    .sort_values(["count_anomalies", "severity_ratio"], ascending=[False, False])
    .loc[:, ["cmdb_id", "log_name", "p95_threshold", "count_anomalies", "earliest_anomaly_timestamp", "max_value_in_window", "severity_ratio"]]
    .head(20)
    .reset_index(drop=True)
)

# 1.8 Distinct samples (up to 50 each)
distinct_log_names = pd.Series(log_p95["log_name"].dropna().unique()[:50], name="distinct_log_name_sample")
distinct_cmdb_ids = pd.Series(log_p95["cmdb_id"].dropna().unique()[:50], name="distinct_cmdb_id_sample")

# --- Part 2: error_logs.csv sample ---

# Create truncated message (first 300 chars)
if "message" in error_logs_df.columns:
    error_logs_df["message_trunc"] = error_logs_df["message"].fillna("").astype(str).str.slice(0,300)
else:
    error_logs_df["message_trunc"] = ""

# Prepare up-to-50-row sample with required columns
if error_logs_df.shape[0] == 0:
    error_logs_sample50 = pd.DataFrame(columns=["timestamp_utc", "cmdb_id", "message_trunc"])
    error_logs_empty_note = "error_logs.csv is empty."
else:
    error_logs_sample50 = error_logs_df.loc[:, ["timestamp_dt", "cmdb_id", "message_trunc"]].head(50).rename(columns={"timestamp_dt": "timestamp_utc"})

# Display compact outputs (variables)
log_result_top20, distinct_log_names, distinct_cmdb_ids, error_logs_sample50

```
Out[3]:
```
Summary (incident window 2024-01-20 12:14:22 → 2024-01-20 12:44:22 UTC):

Log metrics (computed global P95 from the full series, then evaluated values in the incident window):

- Distinct log_name sample: log.error_count, log.row_count
- Distinct cmdb_id sample (up to 50): carts, carts-db, catalogue, front-end, orders, payment, queue-master, session-db, shipping, user, user-db

Top findings (ranked by count_anomalies):
- Several services have log.error_count with p95 = 0 and count_anomalies = 25 (earliest anomaly 2024-01-20T12:17:00Z, max in-window = 0). Affected cmdb_ids include: catalogue, front-end, orders, payment, queue-master, shipping, user, carts, carts-db, session-db (various counts shown). Note: p95 = 0 yields undefined severity_ratio — these zero baselines should be inspected separately.
- carts-db / log.row_count — p95 = 1.0, count_anomalies = 24, earliest = 2024-01-20T12:17:00Z, max = 1, severity = 1.00
- session-db / log.row_count — p95 = 5.0, count_anomalies = 4, earliest = 2024-01-20T12:21:00Z, max = 5, severity = 1.00
- carts / log.row_count — p95 = 125.0, count_anomalies = 3, earliest = 2024-01-20T12:18:00Z, max = 125, severity = 1.00
- user / log.row_count — p95 = 617.6, count_anomalies = 2, earliest = 2024-01-20T12:19:00Z, max = 639, severity ≈ 1.0347
- front-end / log.row_count — p95 = 2063.4, count_anomalies = 2, earliest = 2024-01-20T12:18:00Z, max = 2110, severity ≈ 1.0226
- catalogue / log.row_count — p95 = 169.8, count_anomalies = 2, earliest = 2024-01-20T12:24:00Z, max = 172, severity ≈ 1.0130
- orders / log.row_count, queue-master / log.row_count, shipping / log.row_count, payment / log.row_count — small counts (2) with severity ratios ≈1.00

Interpretation / notes:
- Many log.error_count entries show p95 = 0 and are flagged as anomalies because value >= 0; however max_in_window is also 0. These are zero-baseline cases and should be reviewed separately (they do not indicate a non-zero spike).
- The meaningful non-zero spikes in log.row_count are small in severity (severity ratios mostly ≈1.0–1.035), with the largest being user.log.row_count (~+3.5% over P95) and front-end.log.row_count (~+2.26%).
- Overall, no large severity outliers in logs beyond small increases for row counts; error_count combos with p95 = 0 require inspection of baseline semantics.

Error logs:
- error_logs.csv contains no rows (empty). No error log samples to show.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  p95_threshold  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0      catalogue  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
1      front-end  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
2         orders  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
3        payment  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
4   queue-master  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
5       shipping  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
6           user  log.error_count            0.0               25  2024-01-20 12:17:00+00:00                    0             NaN
7       carts-db    log.row_count            1.0               24  2024-01-20 12:17:00+00:00                    1        1.000000
8       carts-db  log.error_count            0.0               24  2024-01-20 12:17:00+00:00                    0             NaN
9          carts  log.error_count            0.0               23  2024-01-20 12:17:00+00:00                    0             NaN
10    session-db    log.row_count            5.0                4  2024-01-20 12:21:00+00:00                    5        1.000000
11    session-db  log.error_count            0.0                4  2024-01-20 12:21:00+00:00                    0             NaN
12         carts    log.row_count          125.0                3  2024-01-20 12:18:00+00:00                  125        1.000000
13          user    log.row_count          617.6                2  2024-01-20 12:19:00+00:00                  639        1.034650
14     front-end    log.row_count         2063.4                2  2024-01-20 12:18:00+00:00                 2110        1.022584
15     catalogue    log.row_count          169.8                2  2024-01-20 12:24:00+00:00                  172        1.012956
16        orders    log.row_count          133.2                2  2024-01-20 12:19:00+00:00                  134        1.006006
17  queue-master    log.row_count          599.4                2  2024-01-20 12:19:00+00:00                  603        1.006006
18      shipping    log.row_count           66.6                2  2024-01-20 12:19:00+00:00                   67        1.006006
19       payment    log.row_count          106.6                2  2024-01-20 12:19:00+00:00                  107        1.003752, 0    log.error_count
1      log.row_count
Name: distinct_log_name_sample, dtype: object, 0            carts
1         carts-db
2        catalogue
3        front-end
4           orders
5          payment
6     queue-master
7       session-db
8         shipping
9             user
10         user-db
Name: distinct_cmdb_id_sample, dtype: object, Empty DataFrame
Columns: [timestamp_utc, cmdb_id, message_trunc]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric_df if available; otherwise load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
if "timestamp_dt" not in metric_df.columns or not pd.api.types.is_datetime64_any_dtype(metric_df["timestamp_dt"]):
    metric_df["timestamp_dt"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)

# Allowed components
allowed = ["carts", "catalogue", "orders", "payment", "user"]

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 12:14:22", utc=True)
end_ts   = pd.to_datetime("2024-01-20 12:44:22", utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series but only for allowed components
p95_df = (
    metric_df[metric_df["cmdb_id"].isin(allowed)]
    .groupby(["cmdb_id", "kpi_name"], dropna=False)["value"]
    .quantile(0.95)
    .reset_index()
    .rename(columns={"value": "p95_threshold"})
)

# 2) Filter metric data to the incident window and allowed components
window_df = metric_df[
    (metric_df["timestamp_dt"] >= start_ts) &
    (metric_df["timestamp_dt"] <= end_ts) &
    (metric_df["cmdb_id"].isin(allowed))
].copy()

# 3) Merge p95 thresholds into window rows to evaluate anomalies per row
# (this will attach the global p95 to each row for its component-kpi)
window_with_p95 = window_df.merge(p95_df, on=["cmdb_id", "kpi_name"], how="left", validate="many_to_one")

# 4) Determine anomaly flag per row (value >= p95_threshold). If p95 is NaN, treat as False.
window_with_p95["is_anomaly"] = window_with_p95["value"].ge(window_with_p95["p95_threshold"]).fillna(False)

# 5) Compute per-group anomaly aggregates from the window rows
# count_anomalies, earliest_anomaly_timestamp (only among anomaly rows), max_value_in_window
anomaly_counts = (
    window_with_p95
    .groupby(["cmdb_id", "kpi_name"], dropna=False)
    .agg(
        count_anomalies=("is_anomaly", "sum"),
        max_value_in_window=("value", "max")
    )
    .reset_index()
)

earliest_anom = (
    window_with_p95[window_with_p95["is_anomaly"]]
    .groupby(["cmdb_id", "kpi_name"], dropna=False)["timestamp_dt"]
    .min()
    .reset_index()
    .rename(columns={"timestamp_dt": "earliest_anomaly_timestamp"})
)

# 6) Merge p95_df with the window aggregates to include combos that had no rows in the window
result = p95_df.merge(anomaly_counts, on=["cmdb_id", "kpi_name"], how="left")
result = result.merge(earliest_anom, on=["cmdb_id", "kpi_name"], how="left")

# Fill NaNs for counts with 0; keep max_value_in_window as NaN if no rows
result["count_anomalies"] = result["count_anomalies"].fillna(0).astype(int)

# 7) Compute severity_ratio (handle p95 == 0 or NaN -> set severity_ratio = NaN)
def sev(row):
    p = row["p95_threshold"]
    m = row["max_value_in_window"]
    if pd.isna(p) or p == 0 or pd.isna(m):
        return np.nan
    return m / p

result["severity_ratio"] = result.apply(sev, axis=1)

# 8) Select columns and sort by count_anomalies desc then severity_ratio desc; limit to top 50
result_selected = result.loc[:, [
    "cmdb_id", "kpi_name", "p95_threshold", "count_anomalies",
    "earliest_anomaly_timestamp", "max_value_in_window", "severity_ratio"
]].sort_values(["count_anomalies", "severity_ratio"], ascending=[False, False]).reset_index(drop=True)

top50 = result_selected.head(50)

# 9) Socket-related anomalies summary for allowed components
socket_anoms = top50[(top50["kpi_name"] == "socket") & (top50["count_anomalies"] > 0)][["cmdb_id", "earliest_anomaly_timestamp", "count_anomalies"]]
if socket_anoms.shape[0] == 0:
    socket_summary = "No socket-related anomalies found for the allowed components in the incident window."
else:
    # compact summary string listing components and earliest timestamps
    socket_summary = "; ".join(
        f"{row.cmdb_id}: earliest {pd.to_datetime(row.earliest_anomaly_timestamp).isoformat()}"
        for row in socket_anoms.itertuples(index=False)
    )

# Display compact outputs
top50, socket_summary

```
Out[4]:
```
Summary (allowed components: carts, catalogue, orders, payment, user; incident window 2024-01-20 12:14:22 → 2024-01-20 12:44:22 UTC):

- Number of component–KPI combos returned: 32.

Top findings (compact highlights from the per-component/kpi table):
- catalogue / socket — p95 = 5.0, anomalies = 25, earliest = 2024-01-20T12:17:00Z, max = 5.0, severity = 1.00
- orders / error — p95 = 0.0, anomalies = 25, earliest = 2024-01-20T12:17:00Z, max = 0.0 (severity undefined for p95=0)
- payment / socket — p95 = 6.0, anomalies = 12, earliest = 2024-01-20T12:30:00Z, max = 6.0, severity = 1.00
- payment / diskio — p95 ≈ 26,391.62, anomalies = 2, earliest = 2024-01-20T12:17:00Z, max ≈ 69,891.94, severity ≈ 2.65 (notable spike)
- orders / cpu — p95 ≈ 1.88325, anomalies = 2, earliest = 2024-01-20T12:28:00Z, max ≈ 4.33458, severity ≈ 2.30 (notable spike)
- Many memory (mem) entries for the allowed components show small deviations (severity ≈ 1.00–1.001).

Socket-related anomalies for the allowed components (components that showed >=1 anomaly on kpi "socket") and earliest anomaly times:
- catalogue: earliest 2024-01-20T12:17:00Z
- payment: earliest 2024-01-20T12:30:00Z
- user: earliest 2024-01-20T12:40:00Z
- carts: earliest 2024-01-20T12:40:00Z
- orders: earliest 2024-01-20T12:29:00Z

Interpretation notes:
- Socket anomalies are present for multiple allowed components; catalogue and payment show the largest counts (catalogue: 25, payment: 12).
- Most socket entries have severity = 1.00 (p95 equals the max observed in-window), meaning values reached their historical 95th percentile but did not greatly exceed it.
- The most significant spikes (severity > 2) are payment.diskio (~2.65) and orders.cpu (~2.30); these deserve further investigation.
- Some KPIs have p95 = 0 (e.g., certain error metrics); severity is undefined in those cases and they should be interpreted carefully (zero baseline).

If you want, I can:
- show the full ranked table (up to 50 rows) as df.head(50), or
- return only combos with severity_ratio > 1.5 for focused investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id kpi_name  p95_threshold  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   catalogue   socket   5.000000e+00               25  2024-01-20 12:17:00+00:00         5.000000e+00        1.000000
1      orders    error   0.000000e+00               25  2024-01-20 12:17:00+00:00         0.000000e+00             NaN
2     payment   socket   6.000000e+00               12  2024-01-20 12:30:00+00:00         6.000000e+00        1.000000
3     payment   diskio   2.639162e+04                2  2024-01-20 12:17:00+00:00         6.989194e+04        2.648262
4      orders      cpu   1.883249e+00                2  2024-01-20 12:28:00+00:00         4.334584e+00        2.301652
..        ...      ...            ...              ...                        ...                  ...             ...
27       user      mem   8.009032e+06                2  2024-01-20 12:39:00+00:00         8.024242e+06        1.001899
28      carts      mem   2.100173e+08                2  2024-01-20 12:40:00+00:00         2.100935e+08        1.000362
29  catalogue      mem   6.113280e+06                2  2024-01-20 12:34:00+00:00         6.115191e+06        1.000313
30    payment      mem   1.038550e+07                2  2024-01-20 12:39:00+00:00         1.038793e+07        1.000234
31     orders      mem   3.290472e+08                2  2024-01-20 12:39:00+00:00         3.290643e+08        1.000052

[32 rows x 7 columns], 'catalogue: earliest 2024-01-20T12:17:00+00:00; payment: earliest 2024-01-20T12:30:00+00:00; user: earliest 2024-01-20T12:40:00+00:00; carts: earliest 2024-01-20T12:40:00+00:00; orders: earliest 2024-01-20T12:29:00+00:00')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```